Jade Muyambo
M.S Data Science
University of Miami 

2026 ASA South Florida Student Data Challenge

In [74]:
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import RepeatedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, StackingRegressor


In [75]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

rmse_scorer = make_scorer(
    lambda yt, yp: np.sqrt(mean_squared_error(yt, yp)),
    greater_is_better=False
)

In [76]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
labels = pd.read_csv("variable_labels.csv")

print("train shape:", train.shape)
print("test shape:", test.shape)
print("labels shape:", labels.shape)

train.head()

train shape: (1000, 97)
test shape: (200, 96)
labels shape: (96, 2)


,Unnamed: 0,LBDHDD_outcome,DR1TKCAL,DR1TPROT,DR1TCARB,DR1TSUGR,DR1TFIBE,DR1TTFAT,DR1TSFAT,DR1TMFAT,...,RIAGENDR,INDFMPIR,DMDMARTZ,RIDAGEYR,RIDRETH1,RIDRETH3,BMXBMI,BMXWAIST,ALQ111,ALQ121
0,1,51.169706,1459,39.95,152.46,50.34,13.6,78.41,18.062,29.440,...,2,2.17,1,72,5,7,34.3,115.1,1,10
1,2,64.129747,2116,82.54,213.12,75.49,12.7,103.08,48.495,23.529,...,2,1.03,1,61,2,2,22.6,77.3,1,7
2,3,43.333345,2417,61.59,242.22,69.35,14.7,104.62,30.182,31.087,...,1,3.31,1,40,3,3,27.9,106.5,1,3
3,4,58.853188,2018,39.99,206.45,50.56,17.2,42.75,12.418,13.011,...,2,3.82,1,74,3,3,36.5,112.2,1,1
4,5,40.239688,2331,89.59,307.36,161.17,20.1,85.28,22.278,30.190,...,1,2.58,2,80,3,3,24.9,102.6,1,8


In [77]:
TARGET = "LBDHDD_outcome"

for df in [train, test]:
    if "Unnamed: 0" in df.columns:
        df.drop(columns=["Unnamed: 0"], inplace=True)
print("After drop train shape:", train.shape)
print("After drop test shape:", test.shape)

X = train.drop(columns=[TARGET])
y = train[TARGET].astype(float)

test_aligned = test.reindex(columns=X.columns)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("test_aligned shape:", test_aligned.shape)

assert TARGET not in test_aligned.columns



After drop train shape: (1000, 96)
After drop test shape: (200, 95)
X shape: (1000, 95)
y shape: (1000,)
test_aligned shape: (200, 95)


In [78]:
# All 95 predictors are numeric in this dataset

numeric_features = X.columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
    ],
    remainder="drop"
)

print("Numeric feautures:", len(numeric_features))

Numeric feautures: 95


In [79]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

rmse_scorer = make_scorer(
    lambda yt, yp: np.sqrt(mean_squared_error(yt, yp)),
    greater_is_better=False
)

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)


In [80]:
baseline = Pipeline(steps=[
    ("prep", preprocess),
    ("model", Ridge())
])

scores = cross_val_score(baseline, X, y, scoring=rmse_scorer, cv=cv, n_jobs=-1)
print("Baseline Ridge CV RMSE:", -scores.mean(), "+/-", scores.std())

Baseline Ridge CV RMSE: 6.05805188599999 +/- 0.343833918982501


In [82]:
hgb = HistGradientBoostingRegressor(random_state=RANDOM_STATE)

hgb_pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("model", hgb)
])

param_dist = {
    "model__learning_rate": np.linspace(0.01,0.2,25),
    "model__max_depth": [2,3,4,5,6, None],
    "model__max_leaf_nodes": [15,31,63,127],
    "model__min_samples_leaf": [10, 20, 30, 50, 80],
    "model__l2_regularization": np.logspace(-6, 1, 25),
    "model__max_bins": [64, 128, 255],
    "model__early_stopping": [True],
    "model__validation_fraction": [0.1, 0.2],
}

search_hgb = RandomizedSearchCV(
    estimator=hgb_pipe,
    param_distributions=param_dist,
    n_iter=80,
    scoring=rmse_scorer,
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=1,
    error_score="raise"
)
search_hgb.fit(X,y)

best_idx = search_hgb.best_index_
best_mean_rmse = -search_hgb.cv_results_["mean_test_score"][best_idx]
best_std_rmse = search_hgb.cv_results_["std_test_score"][best_idx]

print(f"Tuned HGB CVRMSE: {best_mean_rmse:.4f} +/- {best_std_rmse:.4f}")
print("Best CV RMSE:", -search_hgb.best_score_)
print("Best params:", search_hgb.best_params_)

best_model = search_hgb.best_estimator_


Fitting 25 folds for each of 80 candidates, totalling 2000 fits
Tuned HGB CVRMSE: 4.8388 +/- 0.2545
Best CV RMSE: 4.8388049739515875
Best params: {'model__validation_fraction': 0.1, 'model__min_samples_leaf': 10, 'model__max_leaf_nodes': 31, 'model__max_depth': 3, 'model__max_bins': 128, 'model__learning_rate': np.float64(0.04958333333333334), 'model__l2_regularization': np.float64(1e-06), 'model__early_stopping': True}


In [83]:
best_model.fit(X, y)

test_pred = best_model.predict(test_aligned)

pred_df = pd.DataFrame({"pred": test_pred})
pred_df.to_csv("pred.csv", index=False)

print("Length of test_pred:", len(test_pred))
print("pred_df shape:", pred_df.shape)
print(pred_df.head())
                        

Length of test_pred: 200
pred_df shape: (200, 1)
        pred
0  50.098819
1  53.865072
2  60.776211
3  48.255296
4  43.054872


In [84]:
hgb_best_pipe = clone(best_model)

ridge_pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("model", Ridge(alpha=1.0))
])

enet_pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("model", ElasticNet(alpha=0.01, l1_ratio=0.2, max_iter=10000, random_state=RANDOM_STATE))
])

et_pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("model", ExtraTreesRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

stack_model = StackingRegressor(
    estimators=[
        ("hgb", hgb_best_pipe),
        ("ridge", ridge_pipe),
        ("enet", enet_pipe),
        ("et", et_pipe)
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=5,           
    n_jobs=-1,
    passthrough=False
)

stack_scores = cross_val_score(stack_model, X, y, scoring=rmse_scorer, cv=cv, n_jobs=1)

print("Tuned HGB CV RMSE:", -search_hgb.best_score_)
print("Stacking CV RMSE:", -stack_scores.mean(), "+/-", stack_scores.std())


Tuned HGB CV RMSE: 4.8388049739515875
Stacking CV RMSE: 4.707736641206233 +/- 0.22589936568970065


In [85]:
hgb_cv_rmse = -search_hgb.best_score_
stack_cv_rmse = -stack_scores.mean()

if stack_cv_rmse < hgb_cv_rmse:
    print("Stacking is better. Using stacking model for final fit.")
    final_model = stack_model
else:
    print("Tuned HGB is still better. Keeping tuned HGB for final fit.")
    final_model = best_model

print("Selected model:", type(final_model))


Stacking is better. Using stacking model for final fit.
Selected model: <class 'sklearn.ensemble._stacking.StackingRegressor'>


In [86]:
final_model.fit(X, y)

final_test_pred = final_model.predict(test_aligned)

pred_final_df = pd.DataFrame({"pred": final_test_pred})
pred_final_df.to_csv("pred.csv", index=False)

print("pred_final_df shape:", pred_final_df.shape)
print(pred_final_df.head())

pred_final_df shape: (200, 1)
        pred
0  47.498772
1  54.741355
2  58.967347
3  46.524964
4  41.414689
